[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C62_Coding_Interview_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境（六步协议 / 时间预算 / 复杂度换算 / 五维评分表）

目标：把「面试怎么答」从一堆经验之谈，变成**几个可以运行、可以断言的小工具**。

本 notebook 你会亲手实现：
1. **环境自检** —— 确认 Python / numpy 可用（本课全程不需要 GPU、不需要联网）
2. **六步答题协议自检器** —— 给一段「你实际做了什么」的动作序列，自动指出漏了哪步、哪两步顺序反了
3. **45 分钟时间预算器** —— 按总时长自动缩放，并给出三个不可妥协的检查点
4. **复杂度速查与经验换算** —— 从 `n` 的量级反推「你被允许用什么算法」
5. **五维评分表的代码化** —— 算出「沉默的最优解」为什么输给「会说话的暴力解」
6. **一道题的六步全流程演示** —— 含「面试官会在这里追问什么」的旁注

> 心智模型：**这个环节考的不是「你会不会这道题」，是「你不会的时候是怎么工作的」。**

## 0 · 环境自检

本课全程只用标准库 + numpy。没有 GPU 依赖、不联网、不下载数据。

In [ ]:
import sys, math, random, time
import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)

assert sys.version_info >= (3, 8), '需要 Python 3.8+'
assert hasattr(np, 'argsort')

# 本课会反复用到的三个标准库模块（都不算「绕过考点」的工具）
import bisect, heapq, itertools
from collections import Counter, defaultdict, deque
print('bisect / heapq / itertools / collections 就位')
print('\n✅ 环境自检通过：本课不需要 GPU、不需要联网。')

## 1 · 六步答题协议自检器

协议本身是一个有序清单。自检器要回答两件事：
**（a）你漏了哪一步？（b）哪两步的顺序反了？**

顺序违规的定义：设规范里 `a` 应在 `b` 之前，但你**首次**做 `b` 的时刻早于首次做 `a` —— 记一次违规 `(b, a)`。

In [ ]:
STEPS = [
    ('clarify',  '① 复述与澄清', '复述题目 + 问 3-5 个约束问题'),
    ('example',  '② 举例走一遍', '手造小例子并口算答案（含一个边界）'),
    ('brute',    '③ 先给暴力解', '说清做法与复杂度 —— 这是安全网'),
    ('optimize', '④ 说优化思路', '指出重复计算在哪 + 目标复杂度 + 等点头'),
    ('code',     '⑤ 写代码',     '边写边讲，先主干后边界'),
    ('test',     '⑥ 测试与复杂度', '跑用例 + 边界 + 报时间/空间复杂度'),
]
RANK = {k: i for i, (k, _, _) in enumerate(STEPS)}

def check_protocol(trace):
    """trace: 你实际做的动作序列（step key 的列表，可重复）。
       返回 (missing, inversions)：缺失的步骤、以及顺序违规对 (先做的, 本该更早的)。"""
    missing = [k for k, _, _ in STEPS if k not in trace]
    first = {}                                   # step -> 首次出现的位置
    for pos, t in enumerate(trace):
        first.setdefault(t, pos)
    inversions = set()
    present = [k for k in RANK if k in first]
    for a in present:
        for b in present:
            # 规范里 a 在 b 之前，但实际 b 先发生 -> 违规
            if RANK[a] < RANK[b] and first[a] > first[b]:
                inversions.add((b, a))
    return missing, sorted(inversions, key=lambda p: (RANK[p[0]], RANK[p[1]]))

for k, name, what in STEPS:
    print(f'{name:<12} {what}')

In [ ]:
# —— 三种典型表现 ——
good = ['clarify', 'example', 'brute', 'optimize', 'code', 'test']
rush = ['code', 'code', 'test']                                        # 听完就写
mixed = ['clarify', 'code', 'example', 'brute', 'optimize', 'test']    # 先写了再补流程

m1, i1 = check_protocol(good)
assert m1 == [] and i1 == []

m2, i2 = check_protocol(rush)
assert set(m2) == {'clarify', 'example', 'brute', 'optimize'}, m2
assert i2 == [], '只做了 code/test 两步，这两步的相对顺序是对的'

m3, i3 = check_protocol(mixed)
assert m3 == []
assert i3 == [('code', 'example'), ('code', 'brute'), ('code', 'optimize')], i3
assert len(i3) == 3

print('good  -> 缺失', m1, '违规', i1)
print('rush  -> 缺失', m2, '  ← 丢掉澄清/举例/暴力解/优化思路四步的分')
print('mixed -> 违规', i3, '  ← 代码写在了三步之前')
print('\n✅ 自检器就位：把模拟面试的录像回放一遍，把动作打成 trace 喂进来。')

## 2 · 45 分钟时间预算器

预算表按比例缩放（30 / 45 / 60 分钟通用），最后一项 `buffer` 吸收取整误差，保证**总和严格等于总时长**。

In [ ]:
BUDGET = [('clarify', 4), ('example', 3), ('brute', 4), ('optimize', 6),
          ('code', 18), ('test', 8), ('buffer', 2)]      # 基准：45 分钟

def budget(total_min=45):
    """按比例缩放到 total_min，最后一项吸收取整误差。"""
    base = sum(m for _, m in BUDGET)
    out, acc = [], 0
    for name, m in BUDGET[:-1]:
        v = int(round(total_min * m / base))
        out.append((name, v)); acc += v
    out.append((BUDGET[-1][0], total_min - acc))
    return out

def cumulative(total_min=45):
    """每一步「应该在第几分钟前结束」。"""
    acc, out = 0, []
    for name, m in budget(total_min):
        acc += m
        out.append((name, acc))
    return out

for total in (30, 45, 60):
    plan = budget(total)
    assert sum(v for _, v in plan) == total, (total, plan)
    print(f'{total} 分钟 :', ' '.join(f'{k}={v}' for k, v in plan))

c45 = dict(cumulative(45))
assert dict(budget(45))['code'] == 18
assert c45['brute'] == 11 and c45['optimize'] == 17 and c45['test'] == 43
print('\n三个不可妥协的检查点（45 分钟制）：')
print(f"  第 {c45['brute']:>2} 分钟 —— 必须已说出暴力解与复杂度")
print(f"  第 {c45['optimize']:>2} 分钟 —— 必须开始写代码（无论写的是哪个解）")
print(f"  第 {c45['code'] + 2:>2} 分钟 —— 必须停止写新代码，转入测试")
print('\n✅ 预算器就位。')

## 3 · 复杂度速查：从 n 的量级反推可用算法

依据：一台现代机器每秒可做 $10^8$ 量级基本操作（C/C++），**Python 因解释开销只有 $10^6$–$10^7$**。
下面统一用 `ops = 1e7` 作为 Python 的预算。

In [ ]:
INF = float('inf')

def _fact(n):
    return math.factorial(int(n)) if n <= 20 else INF

def _exp2(n):
    return 2.0 ** n if n <= 1024 else INF

WORK = {                                        # 复杂度名 -> 工作量估计函数
    'log n':   lambda n: math.log2(max(n, 2)),
    'n':       lambda n: float(n),
    'n log n': lambda n: n * math.log2(max(n, 2)),
    'n^2':     lambda n: float(n) ** 2,
    'n^3':     lambda n: float(n) ** 3,
    '2^n':     _exp2,
    'n!':      _fact,
}
ORDER = ['log n', 'n', 'n log n', 'n^2', 'n^3', '2^n', 'n!']   # 由便宜到昂贵

def feasible(n, kind, ops=1e7):
    return WORK[kind](n) <= ops

def suggest(n, ops=1e7):
    """返回在预算内「最昂贵仍可行」的复杂度 —— 它就是你被允许用的算法档次。"""
    ok = [k for k in ORDER if feasible(n, k, ops)]
    return ok[-1] if ok else None

assert feasible(5 * 10**5, 'n log n') and not feasible(10**6, 'n log n')   # 1e6·log2(1e6)≈2e7 > 1e7
assert feasible(10**6, 'n') and not feasible(10**6, 'n^2')
assert feasible(1000, 'n^2') and not feasible(10**4, 'n^2')
assert feasible(20, '2^n') and not feasible(25, '2^n')
assert feasible(10, 'n!') and not feasible(11, 'n!')
assert suggest(10**5) == 'n log n'
assert suggest(10**6) == 'n'                      # Python 预算下 n log n 已经超了
assert suggest(10**6, ops=1e8) == 'n log n'       # C/C++ 预算下还能跑
assert suggest(1000) == 'n^2'
assert suggest(20) == '2^n'
assert suggest(10) == 'n!'

HINT = {'n!': '全排列 / 回溯', '2^n': '状压 DP / 子集枚举', 'n^3': '三重循环 / Floyd',
        'n^2': '双重循环 / 区间 DP', 'n log n': '排序 或 二分 或 堆', 'n': '双指针 / 滑窗 / 哈希'}
print(f"{'n':>10} | {'C/C++ (1e8)':<10} | {'Python (1e7)':<12} | 面试里该说的话")
print('-' * 82)
for n in (10, 20, 500, 3000, 10**5, 10**6, 10**7):
    s7, s8 = suggest(n), suggest(n, ops=1e8)
    flag = '  <- 唯一有分歧的一格' if s7 != s8 else ''
    print(f'{n:>10} | {s8:<10} | {s7:<12} | 「目标 O({s8})，通常意味着 {HINT[s8]}」{flag}')

In [ ]:
# 常数不是不存在的：同为 O(n)，Python 循环与 numpy 向量化差两个数量级
N = 2_000_000
a = np.random.default_rng(0).random(N)

t0 = time.perf_counter(); s1 = float(a.sum());            t_np = time.perf_counter() - t0
t0 = time.perf_counter(); s2 = 0.0
for x in a[:200_000]:                                      # 只跑 1/10，否则太慢
    s2 += float(x)
t_py = (time.perf_counter() - t0) * 10                     # 折算到同样 N

assert abs(s1 - a.sum()) < 1e-6
print(f'numpy  求和 {N} 个数: {t_np*1000:8.2f} ms')
print(f'Python 循环（折算）  : {t_py*1000:8.2f} ms   ← 约 {t_py/max(t_np,1e-9):.0f} 倍')
print('\n✅ 结论：渐近复杂度相同，实际耗时可以差两个数量级。')
print('   面试里说「都是 O(n) 所以一样快」会被追问；说「同为 O(n)，但 Python 循环的常数大约是')
print('   向量化的 20-100 倍，所以真要上车我会向量化」才是满分答案。')

## 4 · 五维评分表：为什么「沉默的最优解」会输

权重是一个可用的近似（各家不同）。重点不是精确预测分数，而是**告诉你时间该花在哪**。

In [ ]:
RUBRIC = [
    ('correctness',   '正确性',   0.30),
    ('complexity',    '复杂度',   0.20),
    ('code_quality',  '代码质量', 0.15),
    ('communication', '沟通',     0.20),
    ('testing',       '测试意识', 0.15),
]
PASS_LINE = 3.5
assert abs(sum(w for _, _, w in RUBRIC) - 1.0) < 1e-12

def score(card):
    """card: {维度: 1-5 分}  ->  加权总分"""
    return sum(w * card[k] for k, _, w in RUBRIC)

A = dict(correctness=5, complexity=5, code_quality=3, communication=1, testing=1)  # 沉默的最优解
B = dict(correctness=4, complexity=3, code_quality=4, communication=5, testing=5)  # 会说话的暴力解

sa, sb = score(A), score(B)
assert abs(sa - 3.30) < 1e-9, sa
assert abs(sb - 4.15) < 1e-9, sb
assert sa < PASS_LINE <= sb

print(f'A 沉默的最优解 : {sa:.2f}   ← 低于及格线 {PASS_LINE}')
print(f'B 会说话的暴力解: {sb:.2f}   ← 强通过')
print()
print(f'沟通 + 测试意识合计权重 = {0.20 + 0.15:.2f}，比正确性的 0.30 还高。')
print('这就是为什么「练边写边讲 + 主动测试」的边际收益高于多刷 50 道题。')

## 5 · 一道题的六步全流程演示

题目（面试官原话，故意含糊）：**「给你一个数组，表示每天的价格。你只能买一次卖一次，求最大收益。」**

下面按六步走一遍，每步都标出**这一步的产出**与**面试官会在这里追问什么**。

In [ ]:
# ── 步骤 ① 复述与澄清 ──────────────────────────────────────────────
RESTATE = '我复述一遍：给一个长度为 n 的价格数组，选 i < j 使 prices[j]-prices[i] 最大，返回这个最大值。'
QUESTIONS = [
    ('必须先买后卖吗？',        '是 —— 所以 i 必须严格小于 j'),
    ('可以不交易吗？',          '可以 —— 那么收益为 0，而不是负数'),
    ('数组可能为空或只有 1 个元素吗？', '可能 —— 返回 0'),
    ('价格会是负数吗？',        '不会 —— 但代码不要依赖这一点'),
    ('n 的量级？',              '最多 1e5 —— 这就排除了 O(n^2)'),
]
print(RESTATE, '\n')
for q, a in QUESTIONS:
    print(f'  Q: {q:<28} A: {a}')

n_max = 10**5
print(f'\n>>> 由 n <= {n_max} 反推：可用的最昂贵复杂度是 O({suggest(n_max)})')
assert suggest(n_max) == 'n log n'
print('    O(n^2) = 1e10 直接出局 —— 这句话在第 1 分钟就该说出来。')
print('\n【面试官旁注】问「可以不交易吗」的人，比不问的人少写一个 bug；')
print('              问「n 的量级」的人，直接拿到复杂度维度的第一分。')

In [ ]:
# ── 步骤 ② 举例走一遍（含边界）────────────────────────────────────
CASES = [
    ([7, 1, 5, 3, 6, 4], 5,  '正常：第 2 天买(1)第 5 天卖(6)'),
    ([7, 6, 4, 3, 1],    0,  '单调下降：一次都不交易'),
    ([],                 0,  '空数组'),
    ([5],                0,  '单元素：买了没法卖'),
    ([3, 3, 3],          0,  '全相同'),
    ([1, 2],             1,  '最短的有效交易'),
]
for arr, want, why in CASES:
    print(f'  {str(arr):<20} -> {want}   ({why})')

print('\n【面试官旁注】第 2-5 条是「边界用例清单」的固定套路：')
print('              空 / 单元素 / 全相同 / 极端单调。四条背下来，每道题都用得上。')

In [ ]:
# ── 步骤 ③ 先给暴力解（安全网）────────────────────────────────────
def max_profit_brute(prices):
    """枚举所有 (买, 卖) 对。时间 O(n^2)，空间 O(1)。"""
    best = 0
    for i in range(len(prices)):
        for j in range(i + 1, len(prices)):
            best = max(best, prices[j] - prices[i])
    return best

for arr, want, _ in CASES:
    assert max_profit_brute(arr) == want, arr

print('暴力解通过全部 6 个用例。口播：')
print('  「最直接的做法是枚举所有买卖对，O(n^2) 时间、O(1) 空间。')
print('   n=1e5 时是 1e10 次操作，跑不完，但它给了我们一个正确的参照。」')
print('\n【面试官旁注】说完这句，「正确性」的下限已经从 1 分抬到 3 分了。')

In [ ]:
# ── 步骤 ④⑤ 优化思路 + 写代码 ─────────────────────────────────────
# 思路：暴力解的内层循环反复在问「i 之前的最小价格是多少」。
#       这个量可以在一次遍历里增量维护 -> 内层循环消失。

def max_profit(prices):
    """时间 O(n)，空间 O(1)。
    循环不变量：处理完下标 i 后，
        min_price = min(prices[0..i])         # 到目前为止见过的最低买入价
        best      = max{prices[j]-prices[k] | k <= j <= i}   # 到目前为止的最优收益
    """
    best = 0
    min_price = float('inf')                 # 空数组时保持 inf，循环不进入，返回 0
    for price in prices:
        # 先用旧的 min_price 结算「今天卖」，保证 买入日 < 卖出日
        best = max(best, price - min_price)
        min_price = min(min_price, price)    # 再更新最低买入价
    return best

for arr, want, why in CASES:
    got = max_profit(arr)
    assert got == want, (arr, got, want)
print('最优解通过全部 6 个用例。')

print('\n【面试官旁注·高频追问】')
print('  Q: 如果把两行调换（先更新 min_price 再结算）会怎样？')
print('  A: 就允许了「同一天买入又卖出」，收益恒 >= 0 不会错，但语义变了；')
print('     若题目改成「必须持有至少一天」，这个顺序就是 bug 的来源。')
print('     —— 能主动说出「这两行的顺序编码了 i < j 这个约束」，是加分点。')

In [ ]:
# ── 步骤 ⑥ 测试与复杂度：随机对拍 + 复杂度陈述 ────────────────────
rng = random.Random(42)
for trial in range(2000):
    n = rng.randint(0, 12)
    arr = [rng.randint(0, 20) for _ in range(n)]
    assert max_profit(arr) == max_profit_brute(arr), arr      # 对拍：最优解 vs 暴力解
print('随机对拍 2000 组通过（长度 0-12，值域 0-20，含大量重复与空数组）。')

# 复杂度的经验验证：n 翻 10 倍，耗时应约翻 10 倍（线性）
def timeit(fn, arr, repeat=3):
    t0 = time.perf_counter()
    for _ in range(repeat):
        fn(arr)
    return (time.perf_counter() - t0) / repeat

big1 = [rng.randint(0, 10**6) for _ in range(100_000)]
big2 = big1 * 10
r = timeit(max_profit, big2) / max(timeit(max_profit, big1), 1e-9)
print(f'n 从 1e5 -> 1e6，耗时比 = {r:.1f}（线性算法应在 8-13 之间）')
assert 5 < r < 25, r

print('\n口播收尾（照抄）：')
print('  「时间 O(n)：每个元素只被访问一次；空间 O(1)：只用了两个标量。')
print('   边界：空数组时 min_price 保持 inf，循环不进入，返回 0；')
print('   单元素时进一次循环，price-inf 为 -inf，best 仍是 0。')
print('   我还用暴力解做了随机对拍，2000 组一致。」')
print('\n✅ 六步走完。注意最后这段话同时拿到了「复杂度」「测试意识」「沟通」三栏的分。')

## ✏️ 练习 1：超时报警器

实现 `budget_alarm(elapsed, done_steps, total=45)`，返回 `'ok'` / `'speed_up'` / `'fallback'`。

规则：
- `expected` = `cumulative(total)` 里**最后一个已完成步骤**的结束时刻（一步都没完成则为 0）
- `elapsed <= expected + 2` → `'ok'`
- `elapsed <= expected + 6` → `'speed_up'`
- 否则 → `'fallback'`（放弃最优解，立刻去写暴力解）

`done_steps` 是已完成步骤的 key 列表，保证是 `STEPS` 的前缀。

In [ ]:
def budget_alarm(elapsed, done_steps, total=45):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# 45 分钟制的累计结束时刻：clarify=4 example=7 brute=11 optimize=17 code=35 test=43
assert budget_alarm(0,  []) == 'ok'
assert budget_alarm(6,  ['clarify']) == 'ok'                      # 4+2
assert budget_alarm(9,  ['clarify']) == 'speed_up'                # <= 4+6
assert budget_alarm(12, ['clarify']) == 'fallback'                # > 10
assert budget_alarm(20, ['clarify', 'example', 'brute', 'optimize']) == 'speed_up'   # 17+2 < 20 <= 17+6
assert budget_alarm(17, ['clarify', 'example', 'brute', 'optimize']) == 'ok'
assert budget_alarm(30, ['clarify', 'example', 'brute', 'optimize']) == 'fallback'
assert budget_alarm(10, ['clarify']) == 'speed_up'                # 45 分钟制：4+6=10 还在窗口内
assert budget_alarm(10, ['clarify'], total=30) == 'fallback'      # 30 分钟制 clarify 只到第 3 分钟 -> 同样 10 分钟已超
for e, d in [(0, []), (6, ['clarify']), (12, ['clarify']), (20, ['clarify', 'example', 'brute', 'optimize'])]:
    print(f'第 {e:>2} 分钟，已完成 {len(d)} 步 -> {budget_alarm(e, d)}')
print('✅ 练习 1 通过：第 17 分钟还没动键盘，报警器应该已经在叫了。')

## ✏️ 练习 2：提分收益排序

实现 `rubric_gap(card)`，返回 `[(维度, 提升到 5 分的加权收益), ...]`：
- 收益 = `权重 × (5 - 当前分)`
- 按收益**降序**；收益相同时按维度名的**字典序升序**（保证结果确定）

In [ ]:
def rubric_gap(card):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
gap = rubric_gap(A)          # A = 沉默的最优解
names = [k for k, _ in gap]
vals = [v for _, v in gap]
assert names == ['communication', 'testing', 'code_quality', 'complexity', 'correctness'], names
assert np.allclose(vals, [0.80, 0.60, 0.30, 0.0, 0.0]), vals
gap_b = rubric_gap(B)
assert gap_b[0][0] == 'complexity' and abs(gap_b[0][1] - 0.40) < 1e-12, gap_b[0]
assert abs(sum(v for _, v in gap) + score(A) - 5.0) < 1e-12, '总收益 + 当前分应恰好等于满分 5'
for k, v in gap:
    print(f'  {k:<15} 提到 5 分可加 {v:.2f}')
print('\n✅ 练习 2 通过：A 该做的不是再刷 100 道题（正确性已满分），是找个人对着讲一遍。')

## ✏️ 练习 3：反解「最大可处理规模」

实现 `max_n(kind, ops=1e7)`：返回满足 `WORK[kind](n) <= ops` 的**最大整数 n**（`n >= 1`），
用二分查找，上界取 `10**9`。（`WORK` 的工作量函数对 n 单调不减，二分是合法的。）

In [ ]:
def max_n(kind, ops=1e7):
    # TODO: 二分 [1, 10**9]
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert max_n('n') == 10_000_000
assert max_n('n^2') == 3162, max_n('n^2')          # 3162^2=9,998,244 <= 1e7 < 3163^2
assert max_n('2^n') == 23, max_n('2^n')            # 2^23=8,388,608 <= 1e7 < 2^24
assert max_n('n!') == 10, max_n('n!')              # 10!=3,628,800 <= 1e7 < 11!
assert max_n('n^3') == 215, max_n('n^3')           # 215^3=9,938,375 <= 1e7 < 216^3
assert 5.2e5 < max_n('n log n') < 5.3e5, max_n('n log n')
assert max_n('n^2', ops=1e8) == 10_000
for k in ORDER:
    print(f'  O({k:<8}) 在 1e7 预算下最多处理 n = {max_n(k):,}')
print('\n✅ 练习 3 通过：这张表反过来读，就是「看到 n 就知道该用什么算法」。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def budget_alarm(elapsed, done_steps, total=45):
    cum = dict(cumulative(total))
    expected = cum[done_steps[-1]] if done_steps else 0
    if elapsed <= expected + 2:
        return 'ok'
    if elapsed <= expected + 6:
        return 'speed_up'
    return 'fallback'

In [ ]:
# 练习 2 参考答案
def rubric_gap(card):
    out = [(k, w * (5 - card[k])) for k, _, w in RUBRIC]
    return sorted(out, key=lambda kv: (-kv[1], kv[0]))

In [ ]:
# 练习 3 参考答案
def max_n(kind, ops=1e7):
    f = WORK[kind]
    lo, hi = 1, 10 ** 9                 # 不变量：f(lo) <= ops < f(hi+1)
    if f(lo) > ops:
        return 0
    while lo < hi:
        mid = (lo + hi + 1) // 2        # 上取整，避免 lo=mid 死循环
        if f(mid) <= ops:
            lo = mid
        else:
            hi = mid - 1
    return lo

---
## 🧪 真实工程胶囊：面试前 24 小时检查单 + 六步口播模板（中英对照）

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 面试前 24 小时检查单
# ══════════════════════════════════════════════════════════════════════
# □ 环境：确认协作编辑器（CoderPad / HackerRank / Google Doc）能打字、能运行
#         Google Doc 形式最坑 —— 没有语法高亮、没有自动缩进，务必提前试一次
# □ 语言：确认面试语言（Python 3）。提前写好这三个 snippet 的肌肉记忆：
#         for i, x in enumerate(a):        l, r = 0, len(a) - 1
#         d[k] = d.get(k, 0) + 1          while l < r: ...
# □ 复习：只复习「模块 01 + 02」的模板（双指针 / 滑窗 / 前缀和 / 二分三模板）
#         不要在面试前一晚学新算法 —— 边际收益为负
# □ 打印：把六步协议 + 三个时间检查点写在一张纸上，放在屏幕旁边
# □ 心态：目标不是「解出来」，是「六步都做到」。第③步做到就有及格分

# ══════════════════════════════════════════════════════════════════════
# B. 六步口播模板（中 / EN —— JD 是英文岗，两套都要能说）
# ══════════════════════════════════════════════════════════════════════
# ① 澄清 CN: 「我先复述一遍确认理解：……。我想确认三件事：输入有序吗？可以有重复吗？n 大概多大？」
#          EN: "Let me restate to make sure I got it: ... Three quick questions:
#               is the input sorted? can there be duplicates? what's the range of n?"
# ② 举例 CN: 「我拿一个小例子走一遍：输入 [..]，我算出来应该是 ..。再看一个边界：空数组返回 0。」
#          EN: "Let me walk through a small example: ... And an edge case: empty input returns 0."
# ③ 暴力 CN: 「最直接的做法是两层循环，O(n^2)。我先说清楚它，这样我们至少有一个正确的参照。」
#          EN: "The brute-force is a double loop, O(n^2). Let me state it first so we have a
#               correct baseline to compare against."
# ④ 优化 CN: 「暴力解里内层循环反复算了同一个量，这个量可以增量维护 -> 目标 O(n)。您觉得这个方向对吗？」
#          EN: "The inner loop recomputes the same quantity; I can maintain it incrementally,
#               which gets us to O(n). Does that direction sound right to you?"
# ⑤ 写码 CN: 「我先写主干，边界稍后补。这个变量记的是『到目前为止见过的最小值』。」
#          EN: "I'll write the main loop first and handle edges after.
#               This variable holds the minimum seen so far."
# ⑥ 收尾 CN: 「时间 O(n)、空间 O(1)。边界我过一遍：空 / 单元素 / 全相同。我还用暴力解对拍了 2000 组。」
#          EN: "Time O(n), space O(1). Edge cases: empty, single element, all-equal.
#               I also cross-checked against the brute force on 2000 random inputs."

# ══════════════════════════════════════════════════════════════════════
# C. 卡住时的三种脱困话术（沉默超过 30 秒是硬扣分）
# ══════════════════════════════════════════════════════════════════════
# 1) 退回上一层: 「我先退一步 —— 暴力解在这里做了什么重复计算？」
#                "Let me step back: what exactly is the brute force recomputing?"
# 2) 举具体例子: 「我拿 [1,1,2] 走一遍，看看我的假设在哪里断掉。」
#                "Let me trace [1,1,2] and see where my assumption breaks."
# 3) 明说要时间: 「我需要 30 秒理一下思路，我在权衡用哈希还是排序。」
#                "Give me 30 seconds — I'm weighing a hash map against sorting."

# ══════════════════════════════════════════════════════════════════════
# D. 与本课程其他部分的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 手撕 IoU / NMS / mAP / 匈牙利 / Focal Loss  -> C61 模块 05（检测专项，本课不重复）
# · 通用算法题（数组/哈希/二分/树图/DP/采样）   -> C62 模块 01-05（本课）
# · ML 系统设计（设计一个 TSR 感知系统）        -> C63
# · 技术知识快问快答（BN vs LN 之类）           -> C64
# · 估算 / 诊断 / 权衡 / 模糊需求               -> C65
'''
print(RECIPE)
for token in ['CoderPad', 'restate', 'brute-force', 'O(n^2)', '30 秒', 'C61 模块 05', 'enumerate']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：环境 / 语言肌肉记忆 / 复习范围 / 中英口播 / 脱困话术 / 课程分工')

### 小结

- **这一环节考的不是「你会不会这道题」，是「你不会的时候是怎么工作的」。**
  交付物有三件：能跑的代码、诚实的复杂度陈述、一套你自己造的测试。多数人只交了第一件。
- **六步协议**（复述澄清 → 举例走一遍 → 先给暴力解 → 说优化思路 → 再写代码 → 测试与复杂度）
  的最大价值在第③步：**先给暴力解把「正确性」的下限从 1 分抬到 3 分**，而且它顺带提供了对拍基准。
- **三个不可妥协的时间检查点（45 分钟制）**：第 11 分钟说出暴力解、
  **第 17 分钟必须开始写代码**、第 37 分钟停止写新代码转入测试。
  超时时要**显式降级并说出来**——「我改为交付暴力解 + 口述复杂度」比默默写不完强得多。
- **沟通(0.20) + 测试意识(0.15) 合计 0.35，比正确性的 0.30 还高。**
  沉默的最优解 3.30 分不及格，会说话的暴力解 4.15 分强通过。
  练「边写边讲 + 主动测试」的边际收益，高于多刷 50 道题。
- **从 n 反推复杂度是免费的提示**：$n\le10^5$ 排除 $O(n^2)$，目标就是 $O(n\log n)$ 或 $O(n)$。
  但也要知道反向用法：$n\le1000$ 时 $O(n^2)$ 就够，此时追最优解是过早优化。
- **Python 内置的判据只有一条**：它就是本题考点 → 不能用；它只是无关工具 → 大方用。
  拿不准就问一句「我可以用 `Counter` 吗，还是您希望我手写？」——两种情况都拿分。

下一站：**模块 01 · 数组与字符串：双指针、滑动窗口、前缀和** ——
ML/CV 岗最高频的题型，也是唯一一个「掌握模板就能覆盖三成题目」的领域。